# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 2: Transformer Architecture - Encoder-only Models</font>

# <font color="#003660">Notebook 3: Text Regression with Transformers</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how to fine-tune a Transformer for a regression task, and <br>
        ... can compare a fine-tuned Transformer with the models from Session 1.
    </font>
</div>
</center>
</p>

In Session 1, we predicted the points of wine reviews with a linear regression on word counts (Notebook 1b), an MLP (Notebook 4), and an LSTM (Notebook 5). All of these models had to learn everything about language from our 10,000 training reviews. In this notebook, we fine-tune a pre-trained Transformer (DistilBERT) on exactly the same task and data. The only difference to text classification (Notebook 2) is the output of the model: instead of one score per class, it predicts a single number.

# Import Packages

As always, we first need to load a number of required Python packages:
- `pandas` and `numpy` for handling data,
- `sklearn` for calculating the prediction error,
- `datasets` and `transformers` from Hugging Face to prepare the data and fine-tune the model,
- `torch` (PyTorch), the deep learning library on which `transformers` is built.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

Check if a GPU is available.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

# Load and Prepare Data

Load the labeled wine reviews and split them into training, validation, and test set. We use exactly the same split as in Session 1 (a sample of 10,000 reviews for training), so that we can compare the results.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/wine_reviews_train.csv")

The points are integers. For a regression model, we convert them to floating point numbers.

In [ ]:
corpus["points"] = corpus["points"].astype(float)

In [ ]:
train_corpus = corpus.iloc[0:80000,].sample(n=10000, random_state=42)
val_corpus = corpus.iloc[80000:90000,]
test_corpus = corpus.iloc[90000:,]

Keep only the review texts and the points, and rename the columns to `text` and `label`, the names the Trainer expects.

In [ ]:
train_corpus = train_corpus[["description", "points"]]
val_corpus = val_corpus[["description", "points"]]
test_corpus = test_corpus[["description", "points"]]

train_corpus = train_corpus.rename(columns={"description": "text", "points": "label"})
val_corpus = val_corpus.rename(columns={"description": "text", "points": "label"})
test_corpus = test_corpus.rename(columns={"description": "text", "points": "label"})

Convert the dataframes into a `DatasetDict` from the Hugging Face Datasets library, as in Notebook 2.

In [ ]:
train_ds = Dataset.from_pandas(train_corpus, preserve_index=False)
val_ds = Dataset.from_pandas(val_corpus, preserve_index=False)
test_ds = Dataset.from_pandas(test_corpus, preserve_index=False)

wine_ds = DatasetDict()

wine_ds['train'] = train_ds
wine_ds['validation'] = val_ds
wine_ds['test'] = test_ds

In [ ]:
wine_ds

# Tokenize Texts

Load the tokenizer of DistilBERT and tokenize all reviews.

In [ ]:
model_name = "distilbert-base-uncased"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

In [ ]:
wine_ds_encoded = wine_ds.map(tokenize, batched=True, batch_size=None)

# Fine-tune Model

For regression, we use the same model class as for classification, `AutoModelForSequenceClassification`, but with a single output neuron (`num_labels = 1`). Setting the problem type to `"regression"` ensures that the model uses a linear activation function for the output layer instead of softmax.

In [ ]:
model = (AutoModelForSequenceClassification
         .from_pretrained(model_name, num_labels = 1)
         .to(device))

model.config.problem_type = "regression"

The ratings are around 88 points, but the freshly initialized output neuron predicts values close to 0. As in Session 1 (Notebooks 4 and 5), we therefore initialize its bias with the mean points of the training set. Without this trick, the model needs many epochs just to move its predictions into the right range: in a test run without it, the test MAE after 3 epochs was 63 points!

In [ ]:
# The ratings are around 88 points, but the freshly initialized output layer predicts values close to 0.
# As in Session 1, we initialize its bias with the mean rating of the training set, so that training
# can focus on learning the differences between wines instead of the offset.
with torch.no_grad():
    model.classifier.bias.fill_(train_corpus["label"].mean())

While the model minimizes the MSE, we monitor the mean absolute error (MAE), the metric we also used in Session 1.

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    mae = np.mean(np.abs(predictions - labels))
    return {"mae": mae}

Configure and start the training, as in Notebook 2. After each epoch, the model is evaluated on the validation set, and at the end, the Trainer loads the weights of the best epoch.

In [ ]:
batch_size = 64
logging_steps = len(wine_ds_encoded["train"]) // batch_size

training_args = TrainingArguments(output_dir="results",
                                  num_train_epochs=3,
                                  learning_rate=3e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  load_best_model_at_end=True,
                                  weight_decay=0.01,
                                  eval_strategy="epoch",
                                  save_strategy="epoch",
                                  disable_tqdm=False,
                                  logging_steps=logging_steps,
                                  report_to="none")

In [ ]:
trainer = Trainer(model=model,
                  args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=wine_ds_encoded["train"],
                  eval_dataset=wine_ds_encoded["validation"])
trainer.train()

# Evaluate Model

Make predictions on the test set and calculate the MAE.

In [ ]:
preds_output = trainer.predict(wine_ds_encoded["test"])

In [ ]:
y_preds = preds_output.predictions

In [ ]:
y_test = wine_ds_encoded["test"]["label"]

In [ ]:
mae = mean_absolute_error(y_test, y_preds)
print(mae)

How does the fine-tuned Transformer compare with the models from Session 1? All models were trained on the same 10,000 reviews and evaluated on the same test set (your numbers may differ slightly, e.g., on a different GPU):

| Model | Test MAE |
|---|---|
| Naive baseline (mean points) | ≈ 2.51 |
| Linear regression on word counts (Notebook 1b) | ≈ 1.58 |
| MLP with learned word embeddings (Notebook 4) | ≈ 1.56 |
| LSTM with learned word embeddings (Notebook 5) | ≈ 1.50 |
| Fine-tuned DistilBERT (this notebook) | ≈ 1.33 |

The pre-trained Transformer clearly beats all models from Session 1. It doesn't have to learn the meaning of words from our 10,000 reviews, because it already learned a lot about language during pre-training on huge amounts of text.

# Exercises

1. **Training time:** Train for more epochs (e.g., 5). Does the validation MAE keep improving?
2. **More data:** Use 40,000 or all 80,000 training reviews (this takes longer, so use a GPU). Does the Transformer benefit as much from more data as the models from Session 1?
3. **Model diagnostics:** Create the diagnostics from Notebook 1b for the Transformer, e.g., the plot of true vs. predicted points. Does the Transformer still suffer from *regression to the mean*?
4. **Your own review:** Write a review of your favorite (or least favorite) wine and let the model predict its points. Hint: tokenize the text with `tokenizer(text, return_tensors="pt").to(device)` and feed it into `model(**inputs)`.

In [ ]:
# YOUR CODE GOES HERE